# Silver Layer — NASA NEO Data Cleaning & Normalization

**Business questions this layer enables:**
- Which near-Earth objects pose the greatest actual risk (closest approach, largest size)?
- How consistent is NASA's reporting of potentially hazardous asteroids over time?

**Input:** the Bronze log (`data/bronze/neo_*.json`) — one immutable, append-only file per day,
produced automatically by the GitHub Actions cron job.

**Data quality issues found in the raw response (this is the "genuinely dirty data" part):**
- Numeric fields (`relative_velocity`, `miss_distance`) arrive as **strings**, not numbers
- `estimated_diameter` is nested in **4 unit systems at once** (km, m, miles, feet) — we keep only km
- Some asteroids have **zero or multiple** `close_approach_data` entries per record
- Backfilled Bronze files can produce **duplicate (asteroid, approach date) pairs**


## 1. Load every Bronze file

In [ ]:
import json
from pathlib import Path

import pandas as pd

BRONZE_DIR = Path("../data/bronze")
SILVER_DIR = Path("../data/silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

bronze_files = sorted(BRONZE_DIR.glob("neo_*.json"))
print(f"Found {len(bronze_files)} bronze files")

raw_records = []
for file_path in bronze_files:
    with open(file_path, "r", encoding="utf-8") as f:
        envelope = json.load(f)
    raw_records.append(envelope)

raw_records[0].keys() if raw_records else None

## 2. Flatten the nested structure

Each envelope wraps NASA's raw response, which nests `near_earth_objects` by date, and each
asteroid nests `estimated_diameter` (4 unit systems) and `close_approach_data` (a list — usually
one entry for a single-day query, but the schema allows more, and sometimes none).

We flatten to **one row per (asteroid, close approach event)**, keeping only kilometer-based units.

In [ ]:
def flatten_bronze_envelope(envelope: dict) -> list[dict]:
    """Flatten one bronze JSON envelope into a list of flat asteroid/approach rows."""
    rows = []
    raw = envelope.get("raw_response", {})
    neo_by_date = raw.get("near_earth_objects", {})

    for query_date, asteroids in neo_by_date.items():
        for asteroid in asteroids:
            diameter = asteroid.get("estimated_diameter", {}).get("kilometers", {})
            approaches = asteroid.get("close_approach_data", [])

            if not approaches:
                # Some records genuinely have no close_approach_data — keep them
                # with null approach fields instead of silently dropping them here
                approaches = [{}]

            for approach in approaches:
                velocity = approach.get("relative_velocity", {})
                distance = approach.get("miss_distance", {})

                rows.append({
                    "neo_id": asteroid.get("id"),
                    "name": asteroid.get("name"),
                    "absolute_magnitude_h": asteroid.get("absolute_magnitude_h"),
                    "estimated_diameter_min_km": diameter.get("estimated_diameter_min"),
                    "estimated_diameter_max_km": diameter.get("estimated_diameter_max"),
                    "is_potentially_hazardous": asteroid.get("is_potentially_hazardous_asteroid"),
                    "is_sentry_object": asteroid.get("is_sentry_object"),
                    "close_approach_date": approach.get("close_approach_date"),
                    "relative_velocity_kph": velocity.get("kilometers_per_hour"),
                    "miss_distance_km": distance.get("kilometers"),
                    "miss_distance_lunar": distance.get("lunar"),
                    "orbiting_body": approach.get("orbiting_body"),
                    "ingested_at_utc": envelope.get("ingested_at_utc"),
                    "query_date": envelope.get("query_date"),
                })

    return rows


all_rows = []
for envelope in raw_records:
    all_rows.extend(flatten_bronze_envelope(envelope))

df = pd.DataFrame(all_rows)
print(f"Flattened to {len(df)} rows")
df.head()

## 3. Type casting — fixing the "numbers stored as strings" problem

Every numeric field from the API arrives as text. We coerce to proper numeric/datetime types
and report how many values fail to parse (`errors="coerce"` turns bad values into `NaN` instead
of crashing, so we can measure data quality instead of guessing).

In [ ]:
numeric_cols = [
    "absolute_magnitude_h",
    "estimated_diameter_min_km",
    "estimated_diameter_max_km",
    "relative_velocity_kph",
    "miss_distance_km",
    "miss_distance_lunar",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["estimated_diameter_mean_km"] = (
    df["estimated_diameter_min_km"] + df["estimated_diameter_max_km"]
) / 2

df["close_approach_date"] = pd.to_datetime(df["close_approach_date"], errors="coerce")
df["ingested_at_utc"] = pd.to_datetime(df["ingested_at_utc"], errors="coerce")

print("Nulls per column after type casting:")
print(df[numeric_cols + ["close_approach_date"]].isna().sum())
print(f"\nDuplicate (neo_id, close_approach_date) pairs: "
      f"{df.duplicated(subset=['neo_id', 'close_approach_date']).sum()}")

## 4. Deduplicate

Backfilling the Bronze log (running the ingestion script manually for past dates) can create
overlapping records if it's ever re-run for the same date. We keep the most recently ingested
version of each (asteroid, approach date) pair.

In [ ]:
before = len(df)
df = df.sort_values("ingested_at_utc").drop_duplicates(
    subset=["neo_id", "close_approach_date"], keep="last"
)
after = len(df)
print(f"Removed {before - after} duplicate rows (kept most recently ingested version)")

## 5. Drop unusable rows and finalize

In [ ]:
before = len(df)
df = df.dropna(subset=["close_approach_date"])
after = len(df)
print(f"Dropped {before - after} rows with no close_approach_date (asteroid had no approach event)")

df = df.reset_index(drop=True)
df.info()

## 6. Save the Silver output

In [ ]:
output_path = SILVER_DIR / "neo_silver.parquet"
df.to_parquet(output_path, index=False)
print(f"Saved {len(df)} clean rows to {output_path}")